# V1DD Functional Stimulus Metrics

Regenerating the `allen_v1dd` stimulus-response metrics — drifting gratings, surround
suppression, natural images, natural movie, receptive fields — from the NWB-Zarr
functional asset.

The original pipeline read a private Isilon HDF5 tree through `OPhysClient`, which no
longer exists; its own README says the code "would not work with the format you have
access to". So this is the same conversion as
**`Functional Data Cell-Cell Correlations.ipynb`**: keep the analysis, replace the data
access.

### Where this notebook currently is

| Milestone | Status |
|---|---|
| **M1 — schema truth** | this notebook |
| **M2 — response engine** | this notebook |
| M3 — natural movie (the deterministic end-to-end check) | not yet |
| M4 — drifting gratings → surround suppression | not yet |
| M5 — natural images / images 12 | not yet |
| M6 — receptive fields | not yet |
| M7 — packaging into the seven published tables | not yet |

M1 and M2 exist to *earn confidence before computing anything*. The per-trial stimulus
table this whole port depends on had never been read off a real file — its schema was
reconstructed from the NWB writer script — so M1 describes what is actually there. M2
proves the response arithmetic against a synthetic trace, where the right answer is known.

Each milestone writes a small JSON to `{save_dir}/checks/`, which is committed, so the
results can be reviewed away from the capsule.

In [1]:
import os
import sys
import time
from os.path import join as pjoin

import numpy as np
import pandas as pd
from IPython.display import display

# Robustly locate utils regardless of the kernel's working directory.
for _candidate in [pjoin("..", "utils"), pjoin("code", "utils"), "utils"]:
    if os.path.isdir(_candidate):
        sys.path.append(os.path.abspath(_candidate))
        break
else:
    raise FileNotFoundError(f"could not locate 'utils'; cwd={os.getcwd()}")

import trial_responses as tr
import v1dd_nwb as vn
from checkpoints import checkpoint
from paths import resolve_data_root, resolve_dataset_dir

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
print(f"numpy {np.__version__} | pandas {pd.__version__}")

numpy 2.5.1 | pandas 2.3.3


## Paths

Input location comes from `utils/paths.py`, so this notebook finds its data on CodeOcean,
off the workshop USB drive, or in a local checkout without editing anything. Outputs
follow the correlations notebook's `scratch`/`results` knob.

In [2]:
mat_version = 1196

data_root = resolve_data_root(f"v1dd_{mat_version}")
functional_dir = resolve_dataset_dir("409828_V1DD_Filtered", root=data_root)

# Where derived tables and verification artifacts go.
output_target = "scratch"   # "scratch" (default, ephemeral) | "results" (reproducible run)
save_dir = pjoin(f"/{output_target}", f"v1dd_{mat_version}_coreg_functional_metrics")
os.makedirs(save_dir, exist_ok=True)

# The two sessions with EM coregistration. The correlations notebook derives this list
# live from CAVE; hard-coding it here keeps M1 free of a network dependency, and the
# schema report prints each session's own (column, volume) so a mismatch is visible.
TARGET_SESSIONS = [(1, "3"), (1, "5")]

print(f"data_root      : {data_root}")
print(f"functional_dir : {functional_dir}")
print(f"save_dir       : {save_dir}")

data_root      : /data/
functional_dir : /data/409828_V1DD_Filtered
save_dir       : /scratch/v1dd_1196_coreg_functional_metrics


## Session discovery

Each session's `(column, volume)` is read from the ROI table *inside* the file, never
from the directory name. If the correlations notebook has already been run, its cached
`session_index.csv` is reused — indexing all 23 sessions from scratch takes about five
minutes.

In [3]:
from pathlib import Path

session_paths = sorted(Path(functional_dir).glob("*/*.nwb.zarr"))
if not session_paths:
    session_paths = sorted(Path(functional_dir).rglob("*.nwb.zarr"))
if not session_paths:
    raise FileNotFoundError(f"no *.nwb.zarr under {functional_dir}")
print(f"{len(session_paths)} session(s) in the asset")

# Reuse the correlations notebook's cached index if it is available.
_cached = resolve_dataset_dir(
    f"v1dd_{mat_version}_coreg_functional_correlation", root=data_root, required=False
)
session_index = None
if _cached and os.path.isfile(pjoin(_cached, "session_index.csv")):
    session_index = pd.read_csv(pjoin(_cached, "session_index.csv"))
    session_index["volume"] = session_index["volume"].astype(str)
    print(f"reusing cached session index from {_cached}")
else:
    print("no cached index found; peeking at each session (~5 min) ...")
    rows = []
    for p in session_paths:
        try:
            nwb, io = vn.open_session(p)
            first = vn.list_planes(nwb)[0]
            rois = nwb.processing[first]["dff"].rois.to_dataframe()
            rows.append({
                "path": str(p), "name": p.parent.name,
                "column": int(pd.to_numeric(rois["column"]).iloc[0]),
                "volume": str(int(pd.to_numeric(rois["volume"]).iloc[0])),
                "n_planes": len(vn.list_planes(nwb)),
                "session_id": str(nwb.session_id),
            })
            io.close()
        except Exception as exc:
            print(f"  could not index {p.parent.name}: {type(exc).__name__}: {exc}")
    session_index = pd.DataFrame(rows)

session_index["target"] = [
    (int(c), str(v)) in TARGET_SESSIONS
    for c, v in zip(session_index["column"], session_index["volume"])
]
targets = session_index.loc[session_index["target"]].reset_index(drop=True)
display(session_index)
print(f"\n{len(targets)} target session(s):")
for _, r in targets.iterrows():
    print(f"  column {r['column']}, volume {r['volume']}  ->  {r['name']}")
if len(targets) != len(TARGET_SESSIONS):
    print(f"\n!! expected {len(TARGET_SESSIONS)} target sessions, found {len(targets)}")

23 session(s) in the asset
reusing cached session index from /data/v1dd_1196_coreg_functional_correlation


,path,name,column,volume,n_planes,session_id,coregistered,target
0,/data/409828_V1DD_Filtered/409828_2018-11-06_1...,409828_2018-11-06_14-02-59_filtered_2026-04-09...,2,1,6,774328450,False,False
1,/data/409828_V1DD_Filtered/409828_2018-11-20_1...,409828_2018-11-20_10-42-45_filtered_2026-04-09...,3,1,6,783110306,False,False
2,/data/409828_V1DD_Filtered/409828_2018-11-21_0...,409828_2018-11-21_09-22-23_filtered_2026-04-16...,4,1,6,783878040,False,False
3,/data/409828_V1DD_Filtered/409828_2018-11-21_1...,409828_2018-11-21_10-56-07_filtered_2026-04-09...,4,2,6,784057573,False,False
4,/data/409828_V1DD_Filtered/409828_2018-11-26_1...,409828_2018-11-26_11-16-25_filtered_2026-04-09...,5,1,6,785378984,False,False
5,/data/409828_V1DD_Filtered/409828_2018-11-27_1...,409828_2018-11-27_11-01-58_filtered_2026-04-09...,2,2,6,785941763,False,False
6,/data/409828_V1DD_Filtered/409828_2018-11-27_1...,409828_2018-11-27_12-29-05_filtered_2026-04-09...,2,3,6,786071018,False,False
7,/data/409828_V1DD_Filtered/409828_2018-11-28_1...,409828_2018-11-28_10-54-56_filtered_2026-04-16...,3,3,6,786879416,False,False
8,/data/409828_V1DD_Filtered/409828_2018-11-29_1...,409828_2018-11-29_13-42-04_filtered_2026-04-09...,5,2,6,788220278,False,False
9,/data/409828_V1DD_Filtered/409828_2018-12-03_1...,409828_2018-12-03_14-25-24_filtered_2026-04-09...,4,3,6,790009715,False,False



2 target session(s):
  column 1, volume 3  ->  409828_2018-12-13_15-10-05_filtered_2026-04-09_05-57-20
  column 1, volume 5  ->  409828_2018-12-14_14-47-35_filtered_2026-04-09_06-13-08


## M1 — schema truth

`schema_report()` describes a session without computing any metric. It **reports rather
than asserts**: a missing table or column is recorded as an error string, because the
point is to learn what the file contains, and a function that raises on the first
surprise tells you much less than one that describes the whole file.

The things this needs to settle:

* Does `intervals['stimulus_table']` have the twelve expected columns? This schema was
  read off the NWB *writer* script and has never been verified against a real file.
  Everything downstream depends on it.
* Do the per-family trial counts match the denominators visible in the published CSVs —
  drifting gratings 8, natural images 8, natural images 12 **40**, natural movie **9**?
* Are there exactly 12 grating directions? The original hard-codes `(dir ± 3) % 12` for
  orthogonal directions and `(dir + 6) % 12` for the null direction, so any other number
  silently computes the wrong metric.
* Is the locally-sparse-noise template a 2× upsample of the 8×14 grid the receptive-field
  code was written against? **This decides whether M6 is possible at all.**
* Is `stop_time - start_time` equal to the 2.0 s the original took from an NWB attribute?
  If not, the response window is a decision rather than a lookup.

In [4]:
%%time
reports = {}
for _, row in targets.iterrows():
    key = f"col{row['column']}_vol{row['volume']}"
    print(f"--- {key}  ({row['name']})")
    nwb, io = vn.open_session(row["path"])
    try:
        reports[key] = vn.schema_report(nwb)
        reports[key]["session"] = {
            "name": row["name"], "session_id": str(row["session_id"]),
            "column": int(row["column"]), "volume": str(row["volume"]),
        }
    finally:
        io.close()

path = checkpoint(
    "schema_report", reports, save_dir,
    sessions=[f"col{r['column']}_vol{r['volume']}" for _, r in targets.iterrows()],
)

--- col1_vol3  (409828_2018-12-13_15-10-05_filtered_2026-04-09_05-57-20)
--- col1_vol5  (409828_2018-12-14_14-47-35_filtered_2026-04-09_06-13-08)
  wrote /scratch/v1dd_1196_coreg_functional_metrics/checks/schema_report.json  (36.0 KB)
CPU times: user 11.9 s, sys: 768 ms, total: 12.7 s
Wall time: 4min 28s


In [5]:
# Compact verdict table -- the questions M1 exists to answer.
EXPECTED_TRIALS = {"drifting_gratings_full": 8, "drifting_gratings_windowed": 8,
                   "natural_images": 8, "natural_images_12": 40, "natural_movie": 9}

rows = []
for key, rep in reports.items():
    st = rep.get("stimulus_table", {})
    ps = rep.get("per_stimulus", {})
    lsn = rep.get("lsn_template", {})
    dgf = ps.get("drifting_gratings_full", {})

    rows.append({"session": key, "question": "stimulus_table present",
                 "answer": "error" not in st, "detail": st.get("error", f"{st.get('n_rows')} rows")})
    rows.append({"session": key, "question": "all 12 expected columns",
                 "answer": st.get("missing_vs_expected") == [],
                 "detail": f"missing={st.get('missing_vs_expected')} extra={st.get('extra_vs_expected')}"})
    rows.append({"session": key, "question": "exactly 12 grating directions",
                 "answer": dgf.get("n_directions") == 12,
                 "detail": str(dgf.get("n_directions"))})
    for fam, want in EXPECTED_TRIALS.items():
        got = (ps.get(fam) or {}).get("n_trials_inferred")
        rows.append({"session": key, "question": f"n_trials[{fam}] == {want}",
                     "answer": got == want, "detail": str(got)})
    rows.append({"session": key, "question": "one spontaneous block",
                 "answer": rep.get("epochs", {}).get("n_spontaneous_blocks") == 1,
                 "detail": str(rep.get("epochs", {}).get("n_spontaneous_blocks"))})
    rows.append({"session": key, "question": "is_soma == pika_conf > 0.5",
                 "answer": all(d.get("is_soma_matches_conf_gt_0.5") is True
                               for d in rep.get("planes", {}).get("detail", {}).values()),
                 "detail": ""})
    rows.append({"session": key, "question": "LSN template reduces to 8x14 (M6 viable)",
                 "answer": lsn.get("rf_viable") is True,
                 "detail": f"{lsn.get('native_shape')} -> {lsn.get('final_shape')}; "
                           f"uniform={lsn.get('blocks_uniform')}; {lsn.get('error', '')}"})
    dur = dgf.get("sweep_duration_s", {})
    rows.append({"session": key, "question": "DG stop-start == 2.0 s",
                 "answer": abs((dur.get("median_stop_minus_start") or 0) - 2.0) < 0.01,
                 "detail": f"stop-start={dur.get('median_stop_minus_start')}, "
                           f"onset-to-onset={dur.get('median_onset_to_onset')}"})

verdict = pd.DataFrame(rows)
display(verdict)
n_bad = int((~verdict["answer"].astype(bool)).sum())
print(f"\n{len(verdict) - n_bad}/{len(verdict)} checks answered as expected")
if n_bad:
    print("\nUnexpected answers -- read these before continuing to M3:")
    display(verdict.loc[~verdict["answer"].astype(bool)])

,session,question,answer,detail
0,col1_vol3,stimulus_table present,True,33214 rows
1,col1_vol3,all 12 expected columns,True,missing=[] extra=[]
2,col1_vol3,exactly 12 grating directions,True,12
3,col1_vol3,n_trials[drifting_gratings_full] == 8,True,8
4,col1_vol3,n_trials[drifting_gratings_windowed] == 8,True,8
5,col1_vol3,n_trials[natural_images] == 8,True,8
6,col1_vol3,n_trials[natural_images_12] == 40,True,40
7,col1_vol3,n_trials[natural_movie] == 9,False,None
8,col1_vol3,one spontaneous block,True,1
9,col1_vol3,is_soma == pika_conf > 0.5,True,



20/24 checks answered as expected

Unexpected answers -- read these before continuing to M3:


,session,question,answer,detail
7,col1_vol3,n_trials[natural_movie] == 9,False,None
11,col1_vol3,DG stop-start == 2.0 s,False,"stop-start=1.9849853515625, onset-to-onset=3.0..."
19,col1_vol5,n_trials[natural_movie] == 9,False,None
23,col1_vol5,DG stop-start == 2.0 s,False,"stop-start=1.9849853515625, onset-to-onset=3.0..."


In [6]:
# The stimulus table itself, for eyeballing. This is the artifact of record for M1.
nwb, io = vn.open_session(targets.iloc[0]["path"])
try:
    stim_table = vn.load_stimulus_table(nwb)
    epochs = vn.epoch_table(nwb)
finally:
    io.close()

print(f"stimulus_table: {stim_table.shape}")
display(stim_table.head(8))
print("\nsweeps per stimulus:")
display(stim_table["stim_name"].value_counts().rename("n_sweeps").to_frame())
print("\nepochs:")
display(epochs)

stimulus_table: (33214, 13)


,id,stim_name,start_time,stop_time,temporal_frequency,spatial_frequency,center_azimuth,center_elevation,direction,frame,image_order,image_index,stimulus_condition_id
0,0,drifting_gratings_full,54.634430,56.619442,1.0,0.04,0.0,0.0,30.0,NaN,NaN,NaN,1
1,1,drifting_gratings_full,57.636951,59.622009,1.0,0.08,0.0,0.0,300.0,NaN,NaN,NaN,22
2,2,drifting_gratings_full,60.639420,62.624439,1.0,0.04,0.0,0.0,300.0,NaN,NaN,NaN,10
3,3,drifting_gratings_full,63.641949,65.626930,1.0,0.08,0.0,0.0,300.0,NaN,NaN,NaN,22
4,4,drifting_gratings_full,66.644447,68.629433,1.0,0.04,0.0,0.0,330.0,NaN,NaN,NaN,11
5,5,drifting_gratings_full,69.646942,71.631927,1.0,0.08,0.0,0.0,120.0,NaN,NaN,NaN,16
6,6,drifting_gratings_full,72.649437,74.634422,1.0,0.08,0.0,0.0,210.0,NaN,NaN,NaN,19
7,7,drifting_gratings_full,75.651909,77.636902,1.0,0.04,0.0,0.0,300.0,NaN,NaN,NaN,10



sweeps per stimulus:


,n_sweeps
stim_name,
natural_movie,29700
locally_sparse_noise,1705
natural_images,944
natural_images_12,480
drifting_gratings_full,192
drifting_gratings_windowed,192
spontaneous,1



epochs:


,id,stim_name,start_time,stop_time,duration
0,0,drifting_gratings_full,54.634430,341.856873,287.222443
1,1,drifting_gratings_windowed,344.876038,632.098572,287.222534
2,2,locally_sparse_noise,635.117737,875.334412,240.216675
3,3,spontaneous,876.318604,1176.552002,300.233398
4,4,natural_images_12,1179.571167,1331.681152,152.109985
5,5,natural_movie,1343.707764,1794.066162,450.358398
6,6,locally_sparse_noise,1798.119507,2098.236084,300.116577
7,7,natural_images,2100.337891,2399.503662,299.165771
8,8,drifting_gratings_windowed,2414.599609,2701.822021,287.222412
9,9,drifting_gratings_full,2704.841309,2992.063721,287.222412


## M2 — response engine

`trial_responses.py` is the arithmetic layer: given traces, timestamps and stimulus onsets,
what was each neuron's mean activity in a window? The original asked this with a Python
loop over every sweep and every bootstrap draw, which costs 40–50 minutes for these two
sessions. Replacing it with a **prefix sum over time** makes each window mean two array
lookups, and the port runs in about five.

The subtlety worth stating: response windows land on a *variable* number of imaging
frames, because stimulus onsets are not frame-aligned. That looks like it forces a loop.
It does not — the samples are never materialised, so `b - a` is just an integer vector.

These checks use a synthetic trace where the right answer is known, so they prove the
engine independently of the data. Two of them are load-bearing:

* **Label-closed windows.** The original selected with `xarray.sel(time=slice(...))`,
  which includes *both* endpoints. A natural `(t >= lo) & (t < hi)` drops one sample per
  trial and shifts every response — the kind of difference that survives into a metric and
  looks like an algorithm bug.
* **Two different window primitives.** Trials use the label-closed form above; the
  bootstrap null uses a *frame-indexed*, fixed-width slice of `round(w / dt)` samples. At
  dt ≈ 0.164 s a 2 s grating window gives 13 samples for a trial and 12 for a null draw.
  That asymmetry is in the original, and reproducing its numbers means reproducing it.

In [7]:
checks = {}
_rng = np.random.default_rng(0)
_n, _dt = 2000, 0.16374
_ts = np.cumsum(_rng.normal(_dt, _dt * 0.002, _n)) + 12.3      # jittered, like a real clock
_traces = _rng.gamma(2.0, 0.5, size=(_n, 7))                   # events-like, non-negative
_starts = _rng.uniform(_ts[5], _ts[-30], size=200)

# 1. label-closed windows, against the definition
_bad = 0
for w0, w1 in [(0.0, 2.0), (-1.0, 0.0), (0.0, 3 * _dt)]:
    a, b = tr.window_bounds(_ts, _starts, w0, w1)
    for i, s in enumerate(_starts):
        want = np.flatnonzero((_ts >= s + w0) & (_ts <= s + w1))   # BOTH ends inclusive
        if not np.array_equal(want, np.arange(a[i], b[i])):
            _bad += 1
checks["window_bounds_label_closed"] = {"mismatches": int(_bad), "n_tested": 600}

a, b = tr.window_bounds(_ts, _starts, 0.0, 2.0)
checks["window_width_varies"] = {"widths": sorted(int(w) for w in np.unique(b - a))}

# 2. prefix-sum means == direct slicing
cs, counts = tr.prefix_sums(_traces)
got = tr.window_means(cs, counts, a, b)
want = np.stack([_traces[a[i]:b[i]].mean(axis=0) for i in range(len(_starts))])
checks["window_means_vs_direct"] = {"max_abs_diff": float(np.max(np.abs(got - want)))}

# 3. nan-aware means
_tn = _traces.copy()
_tn[_rng.random(_tn.shape) < 0.02] = np.nan
csn, cn = tr.prefix_sums(_tn)
gotn = tr.window_means(csn, cn, a, b)
wantn = np.stack([np.nanmean(_tn[a[i]:b[i]], axis=0) for i in range(len(_starts))])
checks["nan_aware_means"] = {"max_abs_diff": float(np.nanmax(np.abs(gotn - wantn))),
                             "counts_array_built": cn is not None}

# 4. bootstrap null: fixed-width frame windows, and reproducible
_null = tr.spontaneous_null(_traces, _ts, _ts[100], _ts[900], (0.0, 2.0),
                            n_boot=500, rng=np.random.default_rng(42))
_again = tr.spontaneous_null(_traces, _ts, _ts[100], _ts[900], (0.0, 2.0),
                             n_boot=500, rng=np.random.default_rng(42))
checks["spontaneous_null"] = {
    "shape": list(_null.shape),
    "reproducible_under_seed": bool(np.array_equal(_null, _again)),
    "null_window_samples": int(round(2.0 / float(np.median(np.diff(_ts))))),
    "trial_window_samples_median": int(np.median(b - a)),
}

# 5. trial scatter, NaN padding, chronological rank
_resp = np.arange(16, dtype=float).reshape(8, 2)
_cond = np.array([0, 1, 2, 0, 1, 2, 0, 2])
_ta = tr.trial_array(_resp, _cond, n_trials=3, n_conditions=3)
checks["trial_array"] = {
    "shape": list(_ta.shape),
    "chronological_within_condition": bool(np.array_equal(_ta[0, :, 0], _resp[[0, 3, 6], 0])),
    "short_condition_nan_padded": bool(np.isnan(_ta[1, 2, 0])),
}

# 6. lifetime sparseness anchors
checks["lifetime_sparseness"] = {
    "uniform_is_0": float(tr.lifetime_sparseness(np.ones((1, 20)))[0]),
    "one_hot_is_1": float(tr.lifetime_sparseness(np.eye(1, 20)) [0]),
}

# 7. frac_trials_above_null excludes NaN trials rather than scoring them
_nl = np.tile(np.linspace(0, 1, 1000), (2, 1))
_trials = np.array([[2.0, 2.0, 2.0, 2.0], [0.5, 2.0, np.nan, np.nan]])
_fr = tr.frac_trials_above_null(_trials, _nl)
checks["frac_trials_above_null"] = {"all_strong": float(_fr[0]), "mixed_with_nan": float(_fr[1])}

ok = (checks["window_bounds_label_closed"]["mismatches"] == 0
      and checks["window_means_vs_direct"]["max_abs_diff"] < 1e-9
      and checks["nan_aware_means"]["max_abs_diff"] < 1e-9
      and checks["spontaneous_null"]["reproducible_under_seed"]
      and checks["trial_array"]["chronological_within_condition"]
      and checks["trial_array"]["short_condition_nan_padded"]
      and abs(checks["lifetime_sparseness"]["uniform_is_0"]) < 1e-12
      and abs(checks["lifetime_sparseness"]["one_hot_is_1"] - 1.0) < 1e-12
      and abs(checks["frac_trials_above_null"]["all_strong"] - 1.0) < 1e-12
      and abs(checks["frac_trials_above_null"]["mixed_with_nan"] - 0.5) < 1e-12)
checks["all_passed"] = bool(ok)

for k, v in checks.items():
    print(f"  {k}: {v}")
print(f"\nengine checks: {'ALL PASSED' if ok else 'FAILED -- do not continue to M3'}")

checkpoint("engine_tests", checks, save_dir, seed=0)

  window_bounds_label_closed: {'mismatches': 0, 'n_tested': 600}
  window_width_varies: {'widths': [12, 13]}
  window_means_vs_direct: {'max_abs_diff': 5.2735593669694936e-14}
  nan_aware_means: {'max_abs_diff': 5.88418203051333e-14, 'counts_array_built': True}
  spontaneous_null: {'shape': [7, 500], 'reproducible_under_seed': True, 'null_window_samples': 12, 'trial_window_samples_median': 12}
  trial_array: {'shape': [3, 3, 2], 'chronological_within_condition': True, 'short_condition_nan_padded': True}
  lifetime_sparseness: {'uniform_is_0': 0.0, 'one_hot_is_1': 1.0}
  frac_trials_above_null: {'all_strong': 1.0, 'mixed_with_nan': 0.5}
  all_passed: True

engine checks: ALL PASSED
  wrote /scratch/v1dd_1196_coreg_functional_metrics/checks/engine_tests.json  (1.0 KB)


'/scratch/v1dd_1196_coreg_functional_metrics/checks/engine_tests.json'

## Next

If the M1 verdict table reads as expected and the engine checks pass, commit
`{save_dir}/checks/` and the next milestone is **M3 — natural movie**.

Natural movie is deliberately first among the metric families. Its
`frac_responsive_trials` is just `mean(response > 0)` — no bootstrap, no threshold, no
stochasticity — so comparing it against the published table exercises the stimulus table,
the trial array, the NaN padding, the response window and the argmax in one shot, with a
hard pass/fail. Everything after it is arithmetic on the same foundation.